# Flower Distributed FL — Orchestrator

This notebook controls a federated learning run across multiple machines.

## Architecture

```
┌─────────────────────────────────────────────────┐
│  SUPERLINK MACHINE  (e.g. 192.168.1.10)         │
│  ┌───────────────────────────────────────────┐  │
│  │  flower-superlink                         │  │
│  │    :9092  ← SuperNodes connect here       │  │
│  │    :9093  ← flwr run connects here        │  │
│  └───────────────────────────────────────────┘  │
└──────────────┬──────────────────────────────────┘
               │  gRPC
   ┌───────────┴────────────┐
   │                        │
┌──┴──────────────┐  ┌──────┴──────────┐
│ SUPERNODE 1     │  │ SUPERNODE 2     │
│ flower-supernode│  │ flower-supernode│
│ (e.g. .11)      │  │ (e.g. .12)      │
└─────────────────┘  └─────────────────┘

  ORCHESTRATOR (this notebook)
  runs `flwr run fl_app/ distributed`
  → sends FAB to SuperLink
  → SuperLink distributes to SuperNodes
  → training & aggregation happen
  → logs stream back here
```

## Before running this notebook

1. On the **SuperLink machine**: `bash start_superlink.sh`
2. On each **SuperNode machine**: `bash start_supernode.sh <SUPERLINK_IP>`
3. Set `SUPERLINK_IP` in the cell below and run all cells.

---
## 0 - Imports & helpers

In [1]:
import importlib
import workshop_helpers

importlib.reload(workshop_helpers)
from workshop_helpers import (
    bin_dir,
    preflight,
    print_port_status,
    run_stream,
    start_background,
    stop_background,
    update_superlink_address,
)

ctx = preflight("fl_app")
flwr_bin_prefix = bin_dir(ctx)

Python executable : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/python
Python version    : 3.13.5
Notebook cwd      : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos
Flower app        : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/fl_app
flwr binary       : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/flwr


flwr            : 1.30.0 @ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/lib/python3.13/site-packages/flwr
numpy           : 2.4.6 @ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/lib/python3.13/site-packages/numpy


The helper functions above keep notebook setup out of the way. The remaining cells show the Flower commands in the same shape students would run from a terminal.

---
## 1 — Configuration

Set `SUPERLINK_IP` to the IP of the machine running `start_superlink.sh`.  
Leave it as `127.0.0.1` if everything runs on the same machine.

In [2]:
SUPERLINK_IP = "127.0.0.1"
SUPERLINK_FLEET_PORT = 9092
SUPERLINK_CONTROL_PORT = 9093

---
## 2 — Quick local simulation (no external machines needed)

Useful to verify that the app code works before going distributed.  
Flower runs 2 virtual SuperNodes in-process using its simulation engine.

In [3]:
command = r"""
flwr run fl_app local \
--run-config num-server-rounds=10 \
--run-config min-clients=2 \
--federation-config num-supernodes=2 \
--stream
""".strip()

rc = run_stream(flwr_bin_prefix + command, cwd=ctx.root)
if rc != 0:
    raise RuntimeError(f"Local simulation failed with exit code {rc}")

$ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/flwr run fl_app local \
  --run-config num-server-rounds=10 \
  --run-config min-clients=2 \
  --federation-config num-supernodes=2 \
  --stream


🎊 Successfully started run 3581698840148843716


INFO :      Starting logstream for run_id `3581698840148843716`


INFO :      Starting Flower Simulation


INFO :      Starting Flower ServerApp, config: num_rounds=10, no round_timeout


...

INFO :      		round 8: 0.045247002502560925


INFO :      		round 9: 0.04138132971497881


INFO :      		round 10: 0.038216237275337184


INFO :      


INFO :      


---
## 3 — Distributed federation

### 3a — Verify connectivity

Make sure the SuperLink is reachable before launching the run.

In [4]:
superlink_ready = print_port_status(
    SUPERLINK_IP,
    {"Fleet API": SUPERLINK_FLEET_PORT, "Control API": SUPERLINK_CONTROL_PORT},
)

Fleet API    127.0.0.1:9092  -> not reachable
Control API  127.0.0.1:9093  -> not reachable


## Distributed Run: Deployment Runtime

This follows Flower's Deployment Runtime flow:

1. Start one `flower-superlink` process.
2. Start two `flower-supernode` processes and pass `--node-config` so each SuperNode gets a distinct partition.
3. Add/use a named SuperLink connection, `local-deployment`, pointing to the SuperLink Control API.
4. Submit the logistic-regression app with `flwr run ... local-deployment --stream`.

For a real deployment, run the SuperLink and SuperNodes on their own machines and use TLS instead of `--insecure`.

In [5]:
SUPERLINK_IP = "127.0.0.1"
SUPERLINK_FLEET_PORT = 9092
SUPERLINK_CONTROL_PORT = 9093
DEPLOYMENT_CONNECTION = "local-deployment"

superlink_ready = print_port_status(
    SUPERLINK_IP,
    {"Fleet API": SUPERLINK_FLEET_PORT, "Control API": SUPERLINK_CONTROL_PORT},
)

command = r"""
flwr run fl_app local-deployment \
--run-config num-server-rounds=10 \
--run-config min-clients=2 \
--stream
""".strip()

if superlink_ready:
    update_superlink_address(f"{SUPERLINK_IP}:{SUPERLINK_CONTROL_PORT}", name=DEPLOYMENT_CONNECTION)
    rc = run_stream(flwr_bin_prefix + command, cwd=ctx.root)
    if rc != 0:
        raise RuntimeError(f"Distributed run failed with exit code {rc}")
else:
    print("SuperLink is not reachable; skipping distributed run.")
    print("Command to run after SuperLink and SuperNodes are started:")
    print(command)

Fleet API    127.0.0.1:9092  -> not reachable
Control API  127.0.0.1:9093  -> not reachable
SuperLink is not reachable; skipping distributed run.
Command to run after SuperLink and SuperNodes are started:
flwr run fl_app local-deployment \
--run-config num-server-rounds=10 \
--run-config min-clients=2 \
--stream


---
## 4 — Optional: run everything locally for testing

If you want to test the distributed path **without multiple machines**,  
this cell starts a SuperLink and two SuperNodes as background subprocesses  
on this machine, waits for them to register, then runs the federation.

In [6]:
RUN_LOCAL_DEPLOYMENT_DEMO = False

superlink_command = r"""
flower-superlink \
--insecure
""".strip()

supernode_0_command = r"""
flower-supernode \
--insecure \\
--superlink 127.0.0.1:9092 \\
--clientappio-api-address 127.0.0.1:9094 \\
--node-config "partition-id=0 num-partitions=2"
""".strip()

supernode_1_command = r"""
flower-supernode \
--insecure \\
--superlink 127.0.0.1:9092 \\
--clientappio-api-address 127.0.0.1:9095 \\
--node-config "partition-id=1 num-partitions=2"
""".strip()

if RUN_LOCAL_DEPLOYMENT_DEMO:
    procs = []
    procs.append(start_background(flwr_bin_prefix + superlink_command, label="SuperLink", cwd=ctx.root))
    procs.append(start_background(flwr_bin_prefix + supernode_0_command, label="SuperNode-0", cwd=ctx.root))
    procs.append(start_background(flwr_bin_prefix + supernode_1_command, label="SuperNode-1", cwd=ctx.root))
    print("Run the distributed cell above, then stop the background processes with:")
    print("stop_background(procs)")
else:
    print("Set RUN_LOCAL_DEPLOYMENT_DEMO = True to start these background commands:")
    for command in [superlink_command, supernode_0_command, supernode_1_command]:
        print()
        print(command)

Set RUN_LOCAL_DEPLOYMENT_DEMO = True to start these background commands:

flower-superlink \
--insecure

flower-supernode \
--insecure \\
--superlink 127.0.0.1:9092 \\
--clientappio-api-address 127.0.0.1:9094 \\
--node-config "partition-id=0 num-partitions=2"

flower-supernode \
--insecure \\
--superlink 127.0.0.1:9092 \\
--clientappio-api-address 127.0.0.1:9095 \\
--node-config "partition-id=1 num-partitions=2"
